# Semana 1: carga y reporte de Fast_Food_Restaurants

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `Fast_Food_Restaurants` desde Hugging Face.
3. Obtener un `DataFrame` de pandas y revisar su estructura.
4. Generar un reporte en pantalla y guardarlo en un archivo `.txt`.

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))



from clases_po import HFDatasetLoader

## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="JasonZhouTI/Fast_Food_Restaurants",
    split="train",
    filename="datos/fast_food_restaurants.csv",
)


## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
%pip install datasets

df = loader.load()
df.head()
df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 14.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 35.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 36.8 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 24.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 [datasets]/26 [datasets]ce-hub]
Note: you may need to restart the kernel to use updated packages.


/workspaces/AD_2026_2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 100%|██████████| 10000/10000 [00:00<00:00, 45368.15 examples/s]


,address,city,country,keys,latitude,longitude,name,postalCode,province,websites
0,324 Main St,Massena,US,us/ny/massena/324mainst/-1161002137,44.92130,-74.89021,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/..."
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,Wendy's,43160,OH,http://www.wendys.com
2,408 Market Square Dr,Maysville,US,us/ky/maysville/408marketsquaredr/1051460804,38.62736,-83.79141,Frisch's Big Boy,41056,KY,"http://www.frischs.com,https://www.frischs.com..."
3,6098 State Highway 37,Massena,US,us/ny/massena/6098statehighway37/-1161002137,44.95008,-74.84553,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/..."
4,139 Columbus Rd,Athens,US,us/oh/athens/139columbusrd/990890980,39.35155,-82.09728,OMG! Rotisserie,45701,OH,"http://www.omgrotisserie.com,http://omgrotisse..."


   ## Transformación mínima
   Se seleccionan solo las columnas `name`, `city`, `province`, `latitude` y `longitude`,
   descartando las demás, para trabajar con un DataFrame más pequeño y enfocado en ubicación.

In [4]:
df_t = df[["name", "city", "province", "latitude", "longitude"]]
df_t.head()

,name,city,province,latitude,longitude
0,McDonald's,Massena,NY,44.92130,-74.89021
1,Wendy's,Washington Court House,OH,39.53255,-83.44526
2,Frisch's Big Boy,Maysville,KY,38.62736,-83.79141
3,McDonald's,Massena,NY,44.95008,-74.84553
4,OMG! Rotisserie,Athens,OH,39.35155,-82.09728


## 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [5]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 10000
Columnas: 10
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   address     10000 non-null  str    
 1   city        10000 non-null  str    
 2   country     10000 non-null  str    
 3   keys        10000 non-null  str    
 4   latitude    10000 non-null  float64
 5   longitude   10000 non-null  float64
 6   name        10000 non-null  str    
 7   postalCode  10000 non-null  str    
 8   province    10000 non-null  str    
 9   websites    9535 non-null   str    
dtypes: float64(2), str(8)
memory usage: 781.4 KB


## 5. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [6]:
from clases_po import Reportes

## 6. Generar y visualizar el reporte como texto

`generar_reporte(df)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en `texto` y se imprime.

In [7]:
reportes = Reportes()
texto = reportes.generar_reporte(df_t)

print(texto)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 5
Duplicados: 1

Columnas y tipos:
- name: str
- city: str
- province: str
- latitude: float64
- longitude: float64

Valores nulos por columna:
- name: 0
- city: 0
- province: 0
- latitude: 0
- longitude: 0

Estadísticas numéricas:
        latitude  longitude
count  10000.000  10000.000
mean      37.552    -91.056
std        5.050     18.556
min       -6.244   -159.379
25%       34.029    -97.413
50%       38.298    -87.661
75%       41.117    -81.458
max       64.840    123.012



## 7. Imprimir el reporte directamente

`imprimir_reporte(df)` ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [8]:
reportes.imprimir_reporte(df_t)

REPORTE DEL DATAFRAME
Filas: 10000
Columnas: 5
Duplicados: 1

Columnas y tipos:
- name: str
- city: str
- province: str
- latitude: float64
- longitude: float64

Valores nulos por columna:
- name: 0
- city: 0
- province: 0
- latitude: 0
- longitude: 0

Estadísticas numéricas:
        latitude  longitude
count  10000.000  10000.000
mean      37.552    -91.056
std        5.050     18.556
min       -6.244   -159.379
25%       34.029    -97.413
50%       38.298    -87.661
75%       41.117    -81.458
max       64.840    123.012


## 8. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en `src/sincronicas/reportes/fast_food_restaurants.txt` cuando se ejecute desde `src/sincronicas`.

In [9]:
ruta = reportes.guardar_reporte(
    df_t,
    "reportes/fast_food_restaurants.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/fast_food_restaurants.txt
